In [ ]:
# установим последнюю версию wandb
!pip install -q --upgrade wandb

In [ ]:
import os
import random
import numpy as np

import wandb

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: masksasha (masksasha-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
run = wandb.init(
    entity="masksasha-hse-university",
    project="insurance-driving-and-damage",
    name="test_wandb_connection",
    config={
        "task": "test",
        "model_type": "no_model_yet",
        "epochs": 5
    }
)

for epoch in range(1, 6):
    acc = epoch / 5
    loss = 1 / epoch + random.random() * 0.05

    run.log({
        "epoch": epoch,
        "accuracy": acc,
        "loss": loss
    })

run.finish()

accuracy,▁▃▅▆█
epoch,▁▃▅▆█
loss,█▄▂▁▁
accuracy,1
epoch,5
loss,0.23202


In [ ]:
seed_everything(CFG.seed)

run = start_wandb_run(
    task_type="image",
    run_name="test_image_tracking",
    model_type="no_model_yet",
    extra_config={
        "dataset": "car_damage_images",
        "target": "damage_class_or_damage_location",
        "comment": "Test run before real image model is ready"
    }
)

for epoch in range(1, 6):
    train_loss = 1 / epoch + random.random() * 0.05
    val_loss = 1 / (epoch + 0.5) + random.random() * 0.05

    accuracy = epoch / 5
    f1 = epoch / 5 - 0.05
    precision = epoch / 5 - 0.03
    recall = epoch / 5 - 0.04

    log_metrics(
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        accuracy=accuracy,
        f1=f1,
        precision=precision,
        recall=recall
    )

wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


accuracy,▁▃▅▆█
epoch,▁▃▅▆█
f1,▁▃▅▆█
precision,▁▃▄▆█
recall,▁▃▅▆█
train_loss,█▄▂▁▁
val_loss,█▄▂▂▁
accuracy,1
epoch,5
f1,0.95
precision,0.97


In [ ]:
ENTITY = "masksasha-hse-university"
PROJECT = "insurance-driving-and-damage"

In [ ]:
run = wandb.init(
    entity=ENTITY,
    project=PROJECT,
    group="tabular_discount_prediction",
    name="test_tabular_wandb_run",
    config={
        "task": "tabular_discount_prediction",
        "model_type": "no_model_yet",
        "dataset": "driving_behavior_mendeley",
        "target": "insurance_discount_or_risk_score",
        "epochs": 5,
        "learning_rate": 0.001
    }
)

for epoch in range(1, 6):
    train_loss = 1 / epoch + random.random() * 0.05
    val_loss = 1 / (epoch + 0.5) + random.random() * 0.05
    mae = 10 / epoch
    rmse = 12 / epoch
    r2 = epoch / 5

    run.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    })

run.finish()

epoch,▁▃▅▆█
mae,█▄▂▁▁
r2,▁▃▅▆█
rmse,█▄▂▁▁
train_loss,█▄▂▁▁
val_loss,█▄▃▂▁
epoch,5
mae,2
r2,1
rmse,2.4
train_loss,0.22848


In [ ]:
class CFG:
    seed = 2026

    num_epochs = 10
    train_batch_size = 64
    val_batch_size = 128
    test_batch_size = 128
    lr = 0.001

    tabular_group = "tabular_discount_prediction"
    image_group = "car_damage_recognition"

In [ ]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ["PYTHONHASHSEED"] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU


def class2dict(f):
    return dict(
        (name, getattr(f, name))
        for name in dir(f)
        if not name.startswith("__")
    )

Функция старта wb-run

In [ ]:
def start_wandb_run(task_type, run_name, model_type, extra_config=None):

    config = class2dict(CFG)

    if extra_config is not None:
        config.update(extra_config)

    config["task_type"] = task_type
    config["model_type"] = model_type

    if task_type == "tabular":
        group = CFG.tabular_group
    elif task_type == "image":
        group = CFG.image_group
    else:
        group = "other"

    run = wandb.init(
        entity=ENTITY,
        project=PROJECT,
        group=group,
        name=run_name,
        config=config,
        reinit=True
    )

    return run

функция логирования метрик

In [ ]:
def log_metrics(epoch=None,
                train_loss=None,
                val_loss=None,
                test_loss=None,
                accuracy=None,
                f1=None,
                precision=None,
                recall=None,
                mae=None,
                rmse=None,
                r2=None,
                map_score=None):

    metrics = {}

    if epoch is not None:
        metrics["epoch"] = epoch

    if train_loss is not None:
        metrics["train_loss"] = train_loss

    if val_loss is not None:
        metrics["val_loss"] = val_loss

    if test_loss is not None:
        metrics["test_loss"] = test_loss

    if accuracy is not None:
        metrics["accuracy"] = accuracy

    if f1 is not None:
        metrics["f1"] = f1

    if precision is not None:
        metrics["precision"] = precision

    if recall is not None:
        metrics["recall"] = recall

    if mae is not None:
        metrics["mae"] = mae

    if rmse is not None:
        metrics["rmse"] = rmse

    if r2 is not None:
        metrics["r2"] = r2

    if map_score is not None:
        metrics["mAP"] = map_score

    wandb.log(metrics)

Функция для отслеживания модели

In [ ]:
# model =
def watch_model(model):
  if model is not None:
    wandb.watch(model, log="all")

Функция сохранения модели как artifact

In [ ]:
def log_model_artifact(model_path, artifact_name, task_type):

    artifact = wandb.Artifact(
        name=artifact_name,
        type="model",
        metadata={
            "task_type": task_type
        }
    )

    artifact.add_file(model_path)
    wandb.log_artifact(artifact)

Функция сохранения датасетов

In [ ]:
def log_dataset_artifact(file_paths, artifact_name, task_type):

    artifact = wandb.Artifact(
        name=artifact_name,
        type="dataset",
        metadata={
            "task_type": task_type
        }
    )

    for file_path in file_paths:
        artifact.add_file(file_path)

    wandb.log_artifact(artifact)